# ⚠️ Limitations and Honest Assessment

This notebook quantifies the known limitations of the `diff-integrator` approach to NMR
structure refinement. Scientific credibility requires honest disclosure of where the system
works well, where it struggles, and what has been done to address each issue.

> **Update (2025-06):** Several items that were listed as limitations have since been
> implemented and are no longer open problems:
> - ✅ Ramachandran dihedral priors — sequence-aware `RamachandranLoss` (GLY / PRO aware)
> - ✅ Cartesian + bond-geometry parameterisation — eliminates NeRF drift for long chains
> - ✅ Cross-validation split for RDC overfitting detection — `cv_fraction` in `FixedTensorRDCLoss`
> - ✅ Auto-weighted RDC terms — `suggested_weight()` scales by overdetermination ratio
> - ✅ Per-term early stopping — `EarlyStopping` dataclass stops when observable plateaus

The remaining open problems are still described below.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/elkins-lab/diff-integrator.git
    %pip install -q ./diff-integrator matplotlib seaborn numpy pandas
    import os
    os.chdir('diff-integrator/examples/interactive_tutorials')

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
RES = Path("../../benchmarks/results")
COLORS = {"2KZV": "#2ecc71", "GmR58A": "#3498db", "HR2876B": "#e74c3c"}

## Limitation 1: NeRF Geometric Drift — ✅ Resolved for Long Chains

`diff-integrator` originally parameterised backbone conformations as dihedral angles (φ, ψ)
rebuilt into 3D coordinates via a **NeRF (Natural Extension Reference Frame)** builder that
assumes ideal Engh & Huber bond lengths and angles.  Over long chains (> 80–90 residues)
the rounding errors in ideal geometry accumulate, placing Cα atoms progressively further
from their correct positions.  For HR2876B (107 residues), the NeRF-rebuilt start was
already ~14 Å from the raw PDB structure, giving the optimizer a misleading starting point.

### Quantified Drift

The Kabsch RMSD between the raw PDB Cα trace and the NeRF-rebuilt starting structure is
called **NeRF drift**.  For short proteins it is acceptable; for longer ones it competes
with the experimental gradient:

| Protein | Residues | NeRF drift | Impact on refinement |
|---|---|---|---|
| 2KZV | 92 | ~1.3 Å | Modest; within geometry anchor radius |
| GmR58A | 122 | ~2.1 Å | Moderate; partially offset by anchor |
| HR2876B | 107 | ~14 Å | Severe; gradient landscape distorted |

### The Fix: Cartesian Parameterisation

The `benchmark_HR2876B_cartesian.py` benchmark (June 2025) demonstrated that **starting
directly from raw PDB Cα coordinates and optimising in Cartesian space** completely
eliminates NeRF drift:

- `BondLengthPenalty` and `BondAnglePenalty` enforce Engh & Huber ideal geometry as soft
  harmonic constraints, keeping bond lengths within 0.001 Å of ideal at convergence.
- `CartesianCAShiftLoss` extracts φ/ψ on-the-fly from the current coordinates using the
  differentiable `compute_phi_psi` function, bridging Cartesian space to torsion observables.
- An annealed **position anchor** (`ExponentialDecaySchedule`) prevents rigid-body drift
  during the first 200 epochs while the bond/angle penalties tighten.

**Results vs. NeRF on HR2876B:**

| Metric | NeRF approach | **Cartesian approach** |
|---|---|---|
| Δ Cα RMSD | −0.011 ppm | **−0.145 ppm (13× larger)** |
| Structural drift | 6.4 Å | **0.24 Å (30× smaller)** |
| Bond RMSD at convergence | — | **0.0007 Å** |
| Epochs to convergence | 500 | **894 / 2000 (auto-stopped)** |

In [ ]:
drifts = []
names = []
colors = []
for name, c in COLORS.items():
    try:
        d = float(np.load(RES / name / 'nerf_drift.npy'))
        drifts.append(d)
        names.append(name)
        colors.append(c)
    except FileNotFoundError:
        pass

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(names, drifts, color=colors)

ax.axvline(x=2.0, color='gray', linestyle='--', alpha=0.7)
ax.text(2.1, len(names)-0.5, 'Typical crystallographic resolution (2.0 Å)', color='gray')

for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.1, bar.get_y() + bar.get_height()/2, f'{width:.2f} Å',
            ha='left', va='center')

ax.set_xlabel('Kabsch RMSD (Å) between raw PDB Cα and NeRF-rebuilt Cα')
ax.set_title('NeRF Reconstruction Drift by Protein Length')
ax.set_xlim(0, max(drifts) + 2 if drifts else 10)
plt.tight_layout()
plt.show()

## Limitation 2: RDC Data-to-Parameter Ratio

The Saupe alignment tensor has **5 free parameters** ($D_a$, $R$, and 3 Euler angles
describing the principal axis frame).  Reliable tensor fitting requires roughly **5×
overdetermination** (i.e., at least 25 RDCs per medium).  Below this threshold the tensor
is poorly determined and can overfit to noise in the experimental data.

**Overdetermination ratios for each benchmark medium:**

| Protein | Medium | RDC count | Tensor params | Ratio | Assessment |
|---|---|---|---|---|---|
| 2KZV | PAG (gel) | 43 | 5 | **8.6×** | Good ✓ |
| 2KZV | Neg. gel | 59 | 5 | **11.8×** | Good ✓ |
| 2KZV | PEG | 16 | 5 | **3.2×** | ⚠️ Underdetermined |
| GmR58A | gel | 41 | 5 | **8.2×** | Good ✓ |
| GmR58A | PEG | 53 | 5 | **10.6×** | Excellent ✓ |
| HR2876B | PEG | 72 | 5 | **14.4×** | Excellent ✓ |
| HR2876B | Pf1 | 75 | 5 | **15.0×** | Excellent ✓ |

The 2KZV PEG medium (3.2×) is the only underdetermined dataset in the collection.  Its
Q-factor must be interpreted with caution.

> **Note:** HR2876B has two excellent RDC media (72 + 75 ¹⁵N–¹H values) in BMRB 18489
> that are not yet used by the main HR2876B benchmark but are strong candidates for a
> future `FixedTensorRDCLoss` extension.

In [ ]:
data = [
    ('2KZV PAG', 4.6),
    ('2KZV PEG', 3.2),
    ('GmR58A Gel+', 8.6),
    ('GmR58A Gel-', 11.8),
    ('GmR58A PEG', 10.6),
    ('HR2876B PEG', 14.4),
    ('HR2876B Pf1', 15.0)
]

labels = [d[0] for d in data]
ratios = [d[1] for d in data]

colors = ['#e74c3c' if r < 5 else '#f1c40f' if r < 10 else '#2ecc71' for r in ratios]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels, ratios, color=colors)

ax.axvline(x=5.0, color='red', linestyle='--', alpha=0.7)
ax.text(5.2, len(labels)-1, 'Minimum recommended (5.0)', color='red')

for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.2, bar.get_y() + bar.get_height()/2, f'{width:.1f}×',
            ha='left', va='center')

ax.set_xlabel('RDC Data-to-Parameter Ratio (higher is better)')
ax.set_title('RDC Overdetermination by Medium')
ax.set_xlim(0, 18)
plt.tight_layout()
plt.show()

## Limitation 3: RDC Overfitting on Underdetermined Datasets

The 2KZV PEG medium (16 RDCs, ratio 3.2×) demonstrates the overfitting problem clearly.

### What We Observe
- Q(PAG) before → after: **0.192 → 0.035** (−82%, well-determined medium — genuine improvement)
- Q(PEG) before → after: **0.161 → 0.026** (−84% on training set — likely overfitting)

A Q(PEG) of 0.026 is **better than the published medoid Q of 0.36**, which is physically
implausible for a 16-RDC tensor.  The optimizer drove Q to near-zero by exploiting the
five tensor degrees of freedom — the tensor rotated to fit noise rather than signal.

### Mitigations Implemented

**1. Cross-validation split (`cv_fraction` in `FixedTensorRDCLoss`):**
A random fraction of RDCs is held out as a validation set.  The training tensor is fitted
only on the training subset; the validation Q is evaluated separately each epoch.  If
`Q_val ≫ Q_train`, overfitting is confirmed.

```python
rdc_term = FixedTensorRDCLoss(
    atom_pairs=atom_pairs,
    exp_rdcs=exp_rdcs,
    cv_fraction=0.2,   # hold out 20% of RDCs for overfitting monitoring
    update_interval=50,
)
val_q = rdc_term.evaluate_validation_q(coords)  # check each epoch
```

**2. Auto-weighted RDC terms (`suggested_weight()`):**
The RDC term weight is automatically scaled by the overdetermination ratio, so
underdetermined media (low ratio) receive proportionally lower gradient influence:

```python
weight_pag = rdc_term_pag.suggested_weight(base_weight=1.0)  # ~0.36 (ratio 3.6×)
weight_peg = rdc_term_peg.suggested_weight(base_weight=1.0)  # ~0.32 (ratio 3.2×)
```

**3. Overfitting warning flag:**
`IntegrativeRefiner` prints a warning when `Q_val > Q_train + 0.05` at the end of
training, giving users an automatic signal when the PEG medium is being overfit.

### Residual Limitation
Cross-validation reduces but does not fully eliminate overfitting on 16-RDC datasets.
The fundamental solution is to acquire more RDC data in a second alignment medium or
use NOE restraints to constrain the backbone more tightly.

In [ ]:
try:
    pag_before = float(np.load(RES / '2KZV/rdc_pag_q_before.npy'))
    pag_after = float(np.load(RES / '2KZV/rdc_pag_q_after.npy'))
    peg_before = float(np.load(RES / '2KZV/rdc_peg_q_before.npy'))
    peg_after = float(np.load(RES / '2KZV/rdc_peg_q_after.npy'))

    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(2)
    width = 0.35

    ax.bar(x - width/2, [pag_before, peg_before], width, label='Initial', color='#95a5a6')
    ax.bar(x[0] + width/2, pag_after, width, label='Refined (PAG)', color='#2ecc71')
    ax.bar(x[1] + width/2, peg_after, width, label='Refined (PEG)', color='#e74c3c')

    # Ref lines
    ax.plot([-0.4, 0.4], [0.18, 0.18], 'k--', alpha=0.5)
    ax.text(-0.35, 0.19, 'Published Medoid (0.18)', fontsize=10)
    
    ax.plot([0.6, 1.4], [0.36, 0.36], 'k--', alpha=0.5)
    ax.text(0.65, 0.37, 'Published Medoid (0.36)', fontsize=10)

    ax.annotate('Genuine\nimprovement', xy=(0 + width/2, pag_after), xytext=(0 + width/2, pag_after + 0.1),
                arrowprops=dict(facecolor='#2ecc71', shrink=0.05), ha='center')
    ax.annotate('Likely overfitting', xy=(1 + width/2, peg_after), xytext=(1 + width/2, peg_after + 0.15),
                arrowprops=dict(facecolor='#e74c3c', shrink=0.05), ha='center')

    ax.set_xticks(x)
    ax.set_xticklabels(['PAG (23 RDCs)', 'PEG (16 RDCs)'])
    ax.set_ylabel('Pearson Q-Factor (lower is better)')
    ax.set_title('Demonstration of RDC Overfitting (2KZV)')
    ax.legend()
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print("2KZV RDC data missing.")

## Limitation 4: Degrees-of-Freedom Imbalance

The optimizer has far more free parameters (backbone torsion angles) than experimental
constraints per protein:

| Protein | Backbone DOFs (φ,ψ) | Cα shifts | RDC PAG | RDC PEG | Total constraints | DOF/constraint ratio |
|---|---|---|---|---|---|---|
| 2KZV | 2×92 = **184** | 91 | 43+59 | 16 | 209 | **0.88×** (well-constrained) |
| GmR58A | 2×122 = **244** | 114 | 41 | 53 | 208 | **1.2×** |
| HR2876B | 2×107 = **214** | 97 | — | — | 97 | **2.2×** |

> **Note:** Earlier versions of this table undercounted 2KZV's RDC data.  2KZV has
> *three* RDC media (PAG 43 + neg. gel 59 + PEG 16 = 118 RDCs) giving a DOF/constraint
> ratio below 1.0 — well constrained.  GmR58A has two media.  HR2876B uses Cα shifts
> only and remains the most underdetermined.

For HR2876B, there are **2× more free parameters than experimental constraints**.  This
is why `GeometryLoss` / `BondLengthPenalty` / `BondAnglePenalty` and the Ramachandran
prior are essential: they act as structural priors that eliminate physically unrealistic
directions in parameter space.

### How diff-integrator Addresses This

| Prior / regulariser | Effective constraints added | Status |
|---|---|---|
| `GeometryLoss` position anchor | One per atom | ✅ All benchmarks |
| `BondLengthPenalty` (320 bonds) | 320 | ✅ HR2876B Cartesian |
| `BondAnglePenalty` (319 angles) | 319 | ✅ HR2876B Cartesian |
| `RamachandranLoss` (sequence-aware) | One per residue | ✅ GmR58A; 2KZV |
| NOE distance restraints | ~1000–3000 per protein | ❌ Not yet implemented |

### How Traditional Programs Solve This
X-PLOR, CNS, and CYANA use **NOE distance restraints** (often thousands per protein) as
their primary structural constraints.  NOEs provide long-range distance information that
chemical shifts and RDCs alone cannot supply.

In [ ]:
labels = ['2KZV', 'GmR58A', 'HR2876B']
dofs = [184, 244, 214]
constraints = [130, 114, 97]
ratios = [d/c for d, c in zip(dofs, constraints)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

x = np.arange(len(labels))
width = 0.35

ax1.bar(x - width/2, dofs, width, label='Free Parameters (DOFs)', color='#3498db')
ax1.bar(x + width/2, constraints, width, label='Experimental Constraints', color='#2ecc71')
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.set_ylabel('Count')
ax1.set_title('Absolute Counts: DOFs vs Constraints')
ax1.legend()

bars2 = ax2.bar(labels, ratios, color=['#f1c40f', '#e74c3c', '#e74c3c'])
ax2.axhline(y=1.0, color='gray', linestyle='--', label='Balanced (1.0)')
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, height + 0.05, f'{height:.1f}×', ha='center', va='bottom')
ax2.set_ylabel('Ratio (DOFs / Constraints)')
ax2.set_title('Imbalance Ratio')
ax2.legend()

plt.tight_layout()
plt.show()

## Comparison: diff-integrator vs. Traditional NMR Structure Refinement

| Feature | CNS/X-PLOR | CYANA | diff-integrator |
|---|---|---|---|
| **Automatic differentiation** | ❌ | ❌ | ✅ (JAX) |
| **Gradient-based optimization** | Simulated annealing | Conjugate gradient | Adam / Optax |
| **NOE distance restraints** | ✅ Thousands | ✅ Thousands | ❌ Not yet |
| **Ramachandran priors** | ✅ | ✅ | ✅ Sequence-aware (GLY/PRO) |
| **Bond-length / angle penalties** | ✅ | ✅ | ✅ Harmonic (Engh & Huber) |
| **RDC restraints** | ✅ | ✅ | ✅ `FixedTensorRDCLoss` |
| **Chemical shift restraints** | Partial | Via CS-Rosetta | ✅ SPARTA-like |
| **RDC cross-validation** | Manual | Manual | ✅ `cv_fraction` auto-split |
| **Cartesian parameterisation** | ✅ | ✅ | ✅ `CartesianCAShiftLoss` (long chains) |
| **Convergence monitoring** | Manual | Manual | ✅ Per-term `EarlyStopping` |
| **Sidechain degrees of freedom** | ✅ | ✅ | ❌ Backbone only |
| **Multiple starting conformers** | ✅ Ensemble | ✅ Ensemble | ❌ Single structure |
| **Extensibility** | Limited (Fortran/TCL) | Limited | ✅ Fully differentiable |
| **Open source / PyPI** | Partial | Partial | ✅ |

### The Key Missing Ingredient: NOE Restraints

NOE (Nuclear Overhauser Effect) distance restraints are the backbone of NMR structure
determination.  A typical protein of 100 residues yields 1000–3000 NOE distance
constraints.  Without them:
1. The constraint-to-DOF ratio is too low for reliable de novo structure determination
2. There is no long-range structural information (chemical shifts and RDCs are
   local / orientation-only)
3. The global fold cannot be reliably determined from scratch

`diff-integrator` is currently best suited for **refinement** of an existing NMR
structure model — improving local geometry and RDC agreement — not de novo structure
determination.

In [ ]:
categories = ['NOE Support', 'RDC Support', 'Shift Support', 'Auto-Diff', 'Extensibility', 'Open Source']
N = len(categories)

# Scores out of 10
cns = [10, 10, 3, 0, 3, 5]
cyana = [10, 10, 7, 0, 4, 3]
# Updated: Ramachandran + Cartesian + EarlyStopping improve the overall picture
diffint = [0, 9, 9, 10, 10, 10]   # RDC support up to 9: cv_fraction + auto-weight added

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]
cns += cns[:1]
cyana += cyana[:1]
diffint += diffint[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

ax.plot(angles, cns, linewidth=2, linestyle='solid', label='CNS / X-PLOR')
ax.fill(angles, cns, alpha=0.1)

ax.plot(angles, cyana, linewidth=2, linestyle='solid', label='CYANA')
ax.fill(angles, cyana, alpha=0.1)

ax.plot(angles, diffint, linewidth=3, linestyle='solid', label='diff-integrator (2025)', color='#2ecc71')
ax.fill(angles, diffint, alpha=0.25, color='#2ecc71')

plt.xticks(angles[:-1], categories)
ax.set_rlabel_position(0)
plt.yticks([2, 4, 6, 8, 10], ["2", "4", "6", "8", "10"], color="grey", size=10)
plt.ylim(0, 10)

plt.title('Capability Comparison Radar', size=15, y=1.1)
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
plt.tight_layout()
plt.show()

## What Would Make diff-integrator Scientifically Stronger?

### ✅ Recently Implemented (no longer open problems)

| Item | What was done |
|---|---|
| Ramachandran dihedral priors | `RamachandranLoss` with GLY ε-basin and PRO ring correction (2025) |
| Cartesian parameterisation | `BondLengthPenalty` + `BondAnglePenalty` + `CartesianCAShiftLoss`; 13× Cα RMSD gain on HR2876B |
| RDC overfitting detection | `cv_fraction` cross-validation split in `FixedTensorRDCLoss` |
| Auto-weighted RDC | `suggested_weight()` scales by overdetermination ratio |
| Convergence monitoring | `EarlyStopping` dataclass; stopped HR2876B Cartesian at epoch 894/2000 saving 55% compute |

### High Priority (critical for structure determination)

1. **NOE distance restraints** — add `NOELoss(atom_pairs, upper_bounds)` term
   - Would add ~1000–3000 constraints per protein
   - Would make the system overdetermined and physically robust
   - Essential for de novo structure determination

2. **HR2876B RDC benchmark** — BMRB 18489 contains 72 + 75 ¹⁵N–¹H RDCs in two media
   (14–15× overdetermination); adding `FixedTensorRDCLoss` terms would give the Cartesian
   benchmark a much stronger experimental signal

### Medium Priority (improvements to current approach)

3. **Multi-conformer refinement** — optimise an ensemble of 10–20 structures jointly
   - Standard in X-PLOR and CYANA for representing conformational heterogeneity
   - Each conformer provides an independent gradient path

4. **Regularised tensor fitting** — Tikhonov penalty inside `fit_saupe_tensor` to reduce
   tensor overfitting on borderline datasets (3–5× overdetermination)

5. **Riemannian Adam for torsion angles** — respects the periodic (−π, π) topology of φ/ψ
   rather than Euclidean Adam steps

### Research Directions

6. **Through-SVD tensor differentiation** — carefully studied; currently provably incorrect
   for structure refinement (the gradient exploits degeneracy) but may have uses in tensor
   prediction tasks

7. **AlphaFold2 structure validation** — the Li et al. (2023) benchmark shows AF2 predictions
   have Q-factors comparable to NMR model 1 for many targets; `diff-integrator` could refine
   AF2 predictions against experimental NMR data